# Operations

**Prerequisites:** `01_what_memory_is.ipynb` and `02_stores.ipynb` from this chapter.

Notebook 2 left four stores behind — a JSON file, two SQLite databases, a Chroma collection, and five edges in a Neo4j instance — each one chosen by what would be asked of it. This notebook is about the verbs.

There are five: **write, update, read, retrieve, forget.** Every store appears to have them, which makes them look like an interface. They are not. Only the *signatures* are portable, and underneath:

- `write` on a graph needs a model call that no other store wants, because a graph does not accept sentences
- `update` on a vector index is a delete and an insert, because there is nothing inside a vector to edit
- `retrieve` does not exist on a dict at all
- `forget` on a graph is usually the wrong operation, and the right one is an end date

Most of this notebook is `retrieve`, because that is where all the difficulty is: ranking more candidates than fit in a prompt, deciding which are worth reading in full, and telling *this memory has not been needed in months* apart from *this memory is no longer true*.

**By the end you'll have:**
- Separated **read** from **retrieve**, and found that only one of the four stores ranks at all
- Timed the same verb across four stores, and hit one that cannot be called from a sentence without a model in the way
- Watched one sentence produce **two** graph edges, and a perfectly good memory produce none
- Proved a vector update is a replacement rather than an edit, by watching the same entry move for one phrase and not another
- Measured where a floor belongs instead of picking one, using questions you know the store cannot answer
- Separated three things that all look like time: reinforcement, validity, and the write date
- Watched the same query return a different third place depending on what the agent had been reading
- Put frequency **inside** lambda rather than beside it, so it stretches the clock instead of competing with recency
- Measured both weights against questions with known answers, and watched one memory take over every query when the reinforcement weight is too high
- Built two-stage recall only after showing it should not exist for one-line memories — and fixed four things in it that looked right and were not
- Given the write path a decision, so `update` has something that calls it and the same fact does not end up in the journal five times
- Separated decay from deletion, made eviction opt-in, and ended on the store where forgetting is neither

## Setup

Everything here opens what notebook 2 left on disk and in Neo4j. Nothing is rebuilt, and the notebook puts back anything it changes, so it can be run twice.

## `read` and `retrieve` are not the same verb

Both hand back a memory, which is why they get treated as one operation with two names. The difference decides how much of this notebook applies to which store.

A **read** is a lookup. You know what you want, you name it, and you get exactly that or nothing.

A **retrieve** is a ranked guess. You cannot name what you want, so you describe it, and what comes back is an ordering over things that might do.

Notebook 2 ran both without separating them:

Two differences, and both matter later.

**A read is exact; a retrieve is ordered.** `timezone` has one right answer and the dict has it. The question about timing out has no right answer stored anywhere — it has entries that resemble it more or less, and the numbers are the *ranking*, not a verdict.

**A read can say no; a retrieve cannot.** `read("shell")` returned `None`. Ask the vector index for three and it returns three, always, whatever is in there. That property is the source of most of the work in this notebook.

Which raises the question of which stores actually retrieve. It is easy to assume all four do something ranked, so it is worth watching the other two answer:

No scores anywhere, and none missing. `Records` filtered and ordered by a column; ask it twice and the same rows come back in the same order, because `ORDER BY at DESC` is not an opinion. `Graph` walked from one node to another and returned the chain it found; there was no second-best path being quietly discarded.

**Only the journal ranks.** The other three answer their questions exactly — a dict by name, SQL by predicate, a graph by traversal — and an exact answer has nothing to rank. That is not a limitation of those three; it is what notebook 2 chose them for.

So from here the notebook splits. `write`, `update` and `forget` are worth walking across all four stores, because each one distorts them differently. `retrieve` is a journal problem, and it takes the rest of the notebook.

## `write`: one verb, four different amounts of work

Same call shape everywhere. Watch what each one costs:

Two local writes in single-digit milliseconds, and one that leaves the machine and takes the better part of a second. The journal write is three writes wearing one name: a row in SQLite, an entry in the FTS5 index, and a vector that only exists once a model has produced it. It is the first place in this chapter where saving a memory can *fail because a service was down*.

The fourth store cannot be written to at all from what we have. `Graph.write` takes a triple:

```python
org.write("Meera", "owns", "billing-api")
```

and what actually arrives is a sentence. *"Devi took over billing-api from Meera on the 20th"* is not a triple, and no amount of string handling turns it into one. Something has to read it and decide what the entities are, which relation holds, and which way it points — and that something is a model.

`03_tools` already built the machinery: put the permitted relations in the schema with `Literal`, and the model picks from them rather than inventing a vocabulary.

Three sentences, three different amounts of graph in them.

**The first one is two edges, not one.** "Took over ... from Meera" starts `Devi owns billing-api` *and* ends `Meera owns billing-api`. A schema with a single `edges` list would have recorded the first and silently lost the second, leaving the graph claiming two owners forever. Splitting `starts` from `ends` is the write path admitting what a graph write actually is: not an insert, a **change**.

**The third one produced nothing at all**, and that is the useful result. *"The pool ran dry under peak traffic"* is a real memory — it is journal entry #1, and notebook 2 retrieved it twice. It just has no entities and no relation, so there is nothing to walk. Not every memory belongs in every store, and this is what that looks like from the inside: the extraction runs, costs a model call, and correctly returns empty.

Which is the real price of a graph write, and it is not the latency. `KeyValue.write` cannot be wrong about what it was given. This one **decides** — what counts as an entity, which of four relations fits, which direction it points — and a decision can be made differently on the next run over the same sentence. Notebook 4 comes back to this, because deciding whether a new memory contradicts an existing one is the same problem one level up.

## `update`, and the store that refuses to have one

An update looks like the least interesting operation here. It is the one where the four stores diverge most.

**`KeyValue` has no separate update**, and does not need one. A key holds one value; writing it again is the update. That is the whole operation.

**`Journal.update` is a delete and an insert wearing one name.** Look at the two rows: the entry barely moved for the phrase both versions still contain, and dropped hard for the one only the original said. That is what tells you the vector was **replaced** rather than adjusted. There is no operation that nudges an embedding to match edited text, because an embedding is not a description of the text — it is the output of a model that read it. Change the text and the only way to get a matching vector is to call the model again. So every update carries the same network cost, and the same chance of failing, as the original write:

```python
def update(self, entry_id, text):
    self.db.execute("UPDATE entries SET text = ? WHERE id = ?", (text, entry_id))
    self.db.execute("INSERT INTO entries_fts (entries_fts) VALUES ('rebuild')")
    self.db.commit()
    self.vectors.update(ids=[str(entry_id)], documents=[text],
                        embeddings=embed([text], task="RETRIEVAL_DOCUMENT"))
```

Three indexes, all rewritten. Miss one and the entry is findable by half the questions that should find it — which is a bug that never raises, and only shows up as a search that quietly stopped working.

**`Graph` has `expire()` instead**, and that is a deliberate refusal. "Meera owns billing-api" did not become *wrong* on the 20th of August; it stopped being *current*. Editing the edge to say Devi would destroy the answer to "who owned this in July", which is exactly the kind of question a graph is kept for. So the edge gets an end date and stays walkable. The forgetting section comes back to this.

**`Records` has no `update()` and should not.** It is an event log: a row says a deploy failed on the 12th, and that stays true forever. What happens next is a *new row*, not an edit to the old one. A store you can update is a store whose history you can lose, and the one thing a run history is for is being the thing you can trust afterwards.

That `DELETE` reaches past the class on purpose — `Records` offers no way to remove a row, for the reason above. It is here because this notebook has to leave the store as it found it, not because deleting events is something you should do.

## `retrieve`, and a journal big enough to need it

Six entries were enough to settle which index answers which question. They cannot show *ranking*, for a reason worth saying out loud: with six memories you can read all six. Ranking only becomes a problem at a size where the answer has to be chosen without anyone looking at everything.

So the rest of this notebook runs against the same agent's journal a few months later — twenty-four entries across four services, from July to the end of November. Still small enough to print, which is the only reason the numbers below can be checked by eye.

### Similarity alone

Notebook 2's question, against the bigger journal. Nothing but cosine, which is all `find_like` does:

A good ranking. `#17` — requests queueing behind a lock while a backfill ran — is the right answer and it is on top.

Two things it cannot do, though, and the rest of this section is those two in order.

1. **It cannot say "I do not have that."** Ask for three and you get three, always.
2. **It has no idea which of these the agent is actually working with.** `#9`, `#14` and `#18` are all plausible; cosine has no way to prefer the one the agent read yesterday.

The first is simpler, so it goes first.

### It always returns k

Ask this journal about something it has never heard of:

Three results each, ranked, with no indication that every one of them is nonsense. A dict answers a missing key with `None`; a vector index has no such answer available to it.

But the scores do know. Line up questions this journal *can* answer against questions it cannot:

**Answerable: 0.72 to 0.78. Unanswerable: 0.52 to 0.55.** A clean gap, so a floor at around `0.6` gives back the property `read` had for free — the ability to return nothing.

One caution, because it is the sort of number that gets copied: `0.6` separates *these* two groups, on *this* journal, with **`gemini-embedding-001`**. Change the model and the band moves. Change the corpus and it moves again. What transfers is the method — ask things you know the answer to, ask things you know it cannot answer, and look at where the two groups land.

### Which of these is the agent actually working with?

Now the second problem. Before any mechanism, the distinction it rests on — because **three different things all look like "time"**, and confusing them is the expensive mistake here.

| The question | What answers it |
|---|---|
| Is this memory **live in what the agent is doing**? | **reinforcement** — how recently and how often it has been *used*. This section. |
| Is this fact **still true**? | **supersession** — replace the memory, or give it a validity range the way `Graph` edges have one. Not a ranking problem at all. |
| **When was it written**? | the `at` column — metadata you filter on, not a ranking signal |

The middle row is the trap. *"What is the current database config?"* feels like a question that wants recency turned up. It does not. The July config entry is not *less relevant* — it is **wrong**, and moving it from second place to fourth still leaves it in the results for the model to read. Wrong facts need deleting or dating, which is the forgetting section. Turning up a weight is not a fix for a false memory.

So the term built here answers only the first row. **It has no opinion about truth.**

Which means it cannot be computed from the `at` column, because being written is not being used. `Journal` records uses separately:

```sql
CREATE TABLE IF NOT EXISTS accesses (
    entry_id INTEGER NOT NULL,
    at       TEXT NOT NULL
);
```

recorded by whatever *consumed* a memory, never by search. Turning up in a top-k is not use; being read is. A memory that keeps surfacing and keeps being ignored should not get to reinforce itself on its own popularity.

Here is the same query against three different working lives:

Look at the bottom two rows. `#18` and `#22` are **0.0016 apart** on raw cosine — as close to tied as this corpus gets. In **A** neither has been touched and `#22` edges it. In **B** the agent has been reading `#18`, and it moves to third while `#22` falls to fifth. In **C** the usage is swapped and so is the order.

Same question, same twenty-four memories, different working life, different answer.

And `#17` and `#9` do not move in any of the three. That is the property that makes this safe to ship: **reinforcement breaks ties, it does not override relevance.**

It is worth seeing exactly how weak that is, because "tiebreaker" is easy to say and hard to picture:

### Frequency belongs inside λ, not beside it

A memory used forty times should not fade at the same rate as one used once. The tempting way to write that is a second term added to the score — and it is wrong, because it puts frequency in **competition** with recency: ten uses last quarter start outranking one use yesterday, and a memory the agent has not touched in months beats the one it read this morning. That is not what "frequently used things are remembered longer" means.

Frequency belongs in the decay rate:

$$\lambda(f) = \frac{\lambda_0}{1 + \alpha f}$$

`λ₀` is the fastest decay, for a memory used once. `f` is the access count. `α` is a **consolidation factor** — how much protection each use buys.

A half-life is just the reciprocal of a decay rate, so the same statement in the form the code uses is:

$$H(f) = H_0 \,(1 + \alpha f), \qquad A = 0.5^{\,t/H(f)}$$

Decay still runs from the **most recent use**. Frequency only sets how fast it runs. One signal says *when did this last matter*, the other *how firmly is it held*, and they multiply rather than trade.

Base `0.5` is not a modelling choice — `0.5^(t/H)` and `e^(-λt)` are the same curve with `λ = ln2/H`. It just makes the knob readable: after `H` days, half as active.

**A hundred uses is not enough.** `#18` trails `#14` by 0.02 of raw cosine, and no amount of reinforcement closes that at `w_reinforcement = 0.2`.

Two things cap it. Activation saturates — once the half-life is long next to the age, `0.5 ** (t/H)` is already near 1 and further uses buy almost nothing (`0.9847` at four uses, `0.9991` at a hundred). And even a perfect activation of 1.0 can only contribute `0.2` to the score.

So the term has a precise amount of power: it can reorder memories within about `0.002` of each other on cosine, and it cannot rescue one that is `0.02` behind however heavily used. If you want a memory to win regardless of how well it matches, reinforcement is the wrong tool and `pinned` is the right one.

Read the bottom block first, because it is the whole argument for putting `f` here. A memory used once is gone in about **six months**. One used sixty times survives **seventeen years**. The tail is long, but *selectively* — earned by use, not handed to everything.

That is the difference from the textbook alternative. ACT-R and the human forgetting literature use a **power law**, which gives every memory a long tail whether or not anyone ever looked at it again. For an agent that is the wrong generosity: forgetting the note nobody reopened is exactly how top-k stays clean. `λ(f)` keeps what got used and lets the rest go.

Now look at the top block sideways. `t = H` gives `0.5` on every row, `t = 2H` gives `0.25`, and so on. The ratio `t/H` is the only thing the formula sees — which means **doubling H is identical to halving t**. Frequency does not change the *shape* of forgetting; it changes how fast the clock runs. Every memory rides the same curve, just at its own speed.

One case this cannot reach, by construction: a memory used **once**, long ago, that genuinely matters. `λ(f)` only knows about usage, so it has nothing to work with — and it should not guess. That case is a `pinned` flag, set deliberately, and the forgetting section builds it.

### Combining the two, and the trap in it

Two numbers, one score. The obvious thing is a weighted sum with an even split. Before writing one, print the terms in separate columns:

**Cosine moves across a range of about 0.12. Activation moves across nearly 0.8.**

A weight is a share of the decision only if the things it multiplies vary by comparable amounts. Written as `0.5 * similarity + 0.5 * activation`, the second term would be doing roughly seven-eighths of the discriminating and the `0.5` in front of the first would be decoration. Cosine over a real corpus lives in a narrow band well away from zero; a decay curve is built to sweep the whole of 0 to 1. They were never on the same scale.

So rescale each term across the candidates before weighting — smallest to 0, largest to 1:

Now a weight means what it says. But look at the `cos_n` column: something scored **1.0000** — and it will, on every query, including the ones about French capitals. Min-max manufactures a perfect score out of whatever it is handed, because the best candidate *defines* the top of the scale.

Which settles an ordering question. **The floor has to run on the raw score, before normalising**, because normalising destroys the only number that could tell an answerable question from an unanswerable one. Filter first, rescale second — the reverse silently makes every query look answered.

### Over-fetching

`recall.py` puts the steps in order: **over-fetch → floor on raw similarity → normalise → weight**.

Over-fetching means asking the index for more candidates than you intend to keep. It is nearly free — one query, the same graph walk — and it matters because the scoring *reorders*. Ask for exactly `k` and score those `k`, and all the scoring can do is shuffle what similarity already chose; a memory ranked seventh on cosine can never reach the top three however live it is:

With `over_fetch=1` the pool *is* the answer — the scoring can reorder those three and nothing else, so a memory sitting fourth on cosine is unreachable no matter what its history is. With `over_fetch=3` nine memories are in play.

The returned three are the same either way, and that is worth stating rather than hiding: **on this corpus, at `w_reinforcement=0.2`, over-fetching never changes the output.** It cannot, for the reason just measured — reinforcement is too weak to promote anything from fourth place.

It stays in because the cost is one query against the same graph, and the case it guards against is real at other settings: raise the reinforcement weight, lengthen the history, or lower `k`, and the pool boundary becomes the thing deciding what you are allowed to see. Paying nothing to remove a hard ceiling is a reasonable trade even when the ceiling is not currently binding.

### Choosing the two weights

Neither weight is a free choice, and they are validated by different tests, because they do different jobs.

**`w_similarity` is validated against questions whose answers you already know.** Six of them, and a sweep:

**An even split gets two of the six wrong.** Everything from `0.6` upward gets all six right. That rules out the number almost everyone starts with, and it is the entire justification for `0.8` — six questions cannot separate `0.8` from `0.9`, so what the sweep establishes is *"similarity must dominate"* and nothing more precise.

Note what this test is blind to. Reinforcement does not change *what the right answer is*; it changes which of several genuinely relevant memories surfaces. A known-answer test cannot see that by construction. So `w_reinforcement` needs its own check, and the thing to check is the failure mode it invites — **rich get richer.** Reinforce whatever you retrieve, and a memory that surfaces gets used, which makes it surface more:

At `0.2` the uses stay spread across more than one memory and first place is where similarity put it. At `0.5` a single memory takes first place on **every** query and collects **every** access in the corpus — including questions it was not the best answer to.

So there are two independent reasons for a small reinforcement weight: the sweep rules out an even split on correctness, and the runaway rules it out on stability. Damping `f` to `ln(1+f)` does *not* prevent this — measured, it produces identical rankings here — because the runaway is a property of the weight, not of the shape of λ.

### Every knob, and how honest each one is

| | Job | How it was set |
|---|---|---|
| `floor = 0.6` | the only step that can return nothing | **measured** — answerable 0.72–0.78 vs unanswerable 0.52–0.55 |
| `w_similarity = 0.8` | how much meaning decides | **measured** — sweep over six known answers |
| `w_reinforcement = 0.2` | how much working life decides | **measured twice** — the sweep, and the runaway |
| `over_fetch = 3` | how many candidates the scoring can reach | reasoned — cheap, and it changes reachability |
| `H₀ = 30 days` | how long a once-used memory stays warm | **guessed** |
| `α = 0.5` | what one use buys in extra warm time | **guessed** |

The last two are worth flagging rather than hiding. A journal spanning five months cannot distinguish a 30-day half-life from a 60-day one — every access is recent compared to either, so `t/H` stays near zero and the curves overlap. The same reason the linear-versus-damped question could not be settled here. Those two numbers need a corpus with years in it, and this one has months.

## Two-stage recall, and when not to build it

Everything so far returns whole memories. That is fine here and it is worth seeing why, because the alternative is a real amount of machinery and most stores should not have it.

A whole retrieve is about two hundred and fifty characters. **A summary of one of these entries would be the entry.** There is nothing to preview, and a function that opened them in full would hand back what you already had.

So: two-stage recall is a solution to a problem this journal does not have. It starts to matter when a memory is not a sentence but a **session** — a debugging run, a postmortem, a design argument — where one memory is a page and three of them will not fit anywhere useful.

Three of those in a prompt is roughly two thousand four hundred characters of history, for one question. It is affordable once and not affordable every turn, and most of it will be irrelevant — the point of a top-3 is that you are not sure which one you want.

So return something small enough to choose from, and fetch the full text only for what gets chosen. The question is where the small thing comes from.

### The summary is written, not derived at read time

Three options, and only one survives contact.

**Truncate.** Free, no model, and the first eighty characters of a debugging session are the paging alert every time. The informative part of a write-up is usually its conclusion.

**Summarise at read time.** A model call per candidate per query, for text that has not changed since it was written. You would pay it again on the next question.

**Summarise at write time.** Once per memory, ever. So the summary is a column, and it is written by whatever writes the text.

That last decision brings an ordering constraint with it, and getting it backwards is a quiet failure rather than a loud one. `Journal` takes a `summarize` hook so the ordering lives in the store rather than in every caller:

```python
def write(self, entries):
    # every summary first: a model call that fails should leave the journal untouched
    summaries = [self.summarize(text) if self.summarize else None for ... in entries]
    for (at, about, source, text), summary in zip(entries, summaries):
        self.db.execute("INSERT INTO entries (..., text, summary) VALUES (...)", ...)
```

The fallible step runs before the durable one, and both columns land together. Summarise *after* writing and a timeout leaves new text sitting behind an old summary — a memory that describes itself incorrectly, which is worse than one with no summary at all.

For the same reason there is no setter for `summary` on its own. A field you can write independently is a field that can drift from what it describes, and the drift is silent: two-stage recall keeps working, just on a description of something the entry no longer says.

Roughly nine to one, and each line says what the memory would let you answer rather than how it opens. Note the third keeps `NOTIFY pgrst,'reload schema'` — the literal somebody would actually search for. A summary that drops the identifiers is a summary you cannot find anything with.

**The bill this creates lands on the write path.** Every save and every update now costs a model call that a single-stage store does not, forever, whether or not that memory is ever searched. Which is one more reason not to build this until memories are genuinely large: on the twenty-four one-line entries above it would be pure cost with nothing bought.

### Two tools, not a pipeline

The obvious shape is a function that returns summaries and then a second call that parses which ids the model wants. The parsing is the problem — the model replies *"let me look at 3 and 7"* and now you own a small, permanently fragile extractor.

Tool calling already solved that. Two tools, and the model's choice arrives as arguments:

Now the loop from chapter 2 runs it, with no memory-specific machinery anywhere — `recall` and `open_memory` are ordinary tools, and `repeat_from` is what lets the model come back for a second turn after reading the summaries:

The model listed, then opened the one it needed, and answered with two facts — the batch size and the recovery time — that appear nowhere in any summary. It was never told about ids, formats or how many to ask for: `open_memory(memory_id=1)` is a tool call, and a tool call is already structured.

**The question was chosen to require the second stage**, which is not a trick so much as the condition for the second stage existing at all. Ask *"why was checkout slow?"* and the summary — *"Backfill job locking orders table caused checkout slowdown; stop job to fix"* — already answers it, so the model answers from the listing and never opens anything. Measured over three runs, it did exactly that. **That is correct behaviour**, and it is worth sitting with: if your summaries are good, most questions never reach the second stage, and the token saving is the whole return on the write-time cost.

One caveat on what you see above, because notebook outputs make things look more certain than they are: it is one run. Ten runs of this cell *without* `EXTRACTORS` went straight through seven times. The other three took a detour — guessed `open_memory(memory_id=0)`, passed a date as the id, or searched again with the question reworded — and all three traced to one place: the listing had reached the model without its `#1`. It looked like a model that could not tell when to stop. It was a model with nothing to open. With `EXTRACTORS`, five runs of five went `recall -> open_memory -> answer`. That is why the cell prints the tool sequence rather than resting on the final message.

Two things in that cell are corrections rather than decoration, and two more in the tools above it. All four were failures before they were fixes.

**The trailing `Observe()`.** `Act` does not put a tool's result into the conversation; it leaves it on `conversation.pending`, and `Observe` is what turns that into something the model can read. Leave it out — as the first version of this cell did — and the model calls `recall`, sees nothing come back, and calls `recall` again with slightly different words until the runtime stops it for repeating itself. The tools were fine. The loop was missing the step that closes it.

**`extractors=EXTRACTORS` on that `Observe`.** By default `Observe` has a model restate each tool result as one plain sentence. For `recall` the rewrite kept the entry but usually dropped the `#1` in front of it — the id survived in about one run in five — and the id is the only thing that makes an entry openable. `EXTRACTORS`, from `memory_tools.py`, hands `recall` and `open_memory` results over word for word, with no model call. A listing is already written for the model to read; there is nothing to restate.

**The date in the listing.** The first version showed `#id  summary` and nothing else, and the model opened the entry about the 14th of October when asked about the 6th of November. The summaries were good — they simply did not say *when*, and there is no way to pick the November one out of a list that never mentions dates.

**And one sentence in a docstring.** With `open_memory` described as *"Read one memory in full."*, the model called `recall` five times in a row and never opened anything. Adding *"Call this before answering from a recall result"* was the whole fix. The docstring is not documentation here — `03_tools` sends it to the API as the tool's `description`, so it is the only instruction the model ever gets about **when** a tool applies. `recall` returning ids is not a hint the model is obliged to take; something has to say that ids are for opening.

That is the same lesson as keeping `PGRST204` in a summary, arriving from the other direction: **the listing is what the choice is made from, so it has to carry whatever the choice depends on.** A summary is a compression of the text; the date was never in the text to begin with, it was in the row. Stage one is a rendering job, not just a summarising one.

**And `open_memory` does a second job that is easy to miss.** It records the access:

```python
sessions.used([memory_id], TODAY)      # being opened is use; being listed is not
```

Which settles a question the reinforcement section left open: *who writes to the `accesses` table?* The rule is that **the last stage records**, and the two arrangements put it in different places.

**Two-stage:** `open_memory` records, and `Recall` is built with `record_use=False`. A memory listed and skipped decays exactly as if it had never come up, because nothing wrote anything down when it was listed. That is the sharp signal — *something read this* — and it is only available because there is a second stage to be the last one.

**Single-stage:** the retrieve is the last stage, so it records what it returned. `Recall` does this by default; there is nothing downstream that could. The signal is coarser — it says *the ranking kept choosing this*, not *something read it* — and for short memories that is a fair trade, because the returned line **is** the whole memory and there was never anything further to read. Being returned is as close to being used as that store can get.

What you must not do is neither. A single-stage store built with `record_use=False` has an `accesses` table that stays empty forever, so every activation falls back to the write date and the reinforcement term quietly degenerates into the write-date decay this section spent its first half arguing against. It keeps working, it keeps producing plausible numbers, and it has stopped measuring what it claims to.

### When not to build it

Worth stating plainly, because everything above is machinery:

| | Two-stage |
|---|---|
| memories are a sentence or two | **no** — the summary is the memory, and `open_memory` returns what you already had |
| memories are sessions, transcripts, documents | **yes** — a top-k does not fit, and choosing is cheap |
| you need usage to be meaningful | **yes** — it is the only place an honest "this was used" signal exists |
| writes are frequent and searches are rare | **think again** — the cost is per write, and it is paid whether or not anything is ever recalled |

The journal from the rest of this notebook stays single-stage, and should.

## Writing without contradicting yourself

`update` arrived early in this notebook as a *mechanism* — change the text, re-embed, rewrite the summary. Nothing has yet decided to call it.

That gap has a specific shape. An agent finishes a run, concludes something, and writes it down. It does not know what it already remembers, so everything it concludes is a new entry. Do that for a month and the journal holds the same fact five times, three of them out of date, and a retrieve returns whichever the ranking happens to prefer.

The failure is not that a wrong memory exists. It is that **both versions exist**, and once they do the store cannot tell you which is current — that information was thrown away at write time, by writing instead of updating.

So a write is not one operation. It is a search, then a decision, then one of three actions:

| | when | effect |
|---|---|---|
| **add** | nothing remembered covers this | a new entry |
| **update** | something says this, with a value this replaces | the old text is **gone** — one entry, current |
| **ignore** | something already says this | nothing |

`update` is what keeps contradictions from accumulating, because it replaces in place rather than writing alongside.

### The decision needs the retrieve half of this notebook

The search that decides a write is the same search that answers a question. To know whether a conclusion is new, you have to find what resembles it — which is `Recall`, pointed at the thing about to be written rather than at a user's question.

Then a model classifies the relationship. `03_tools` supplies the shape: `Literal` puts the three actions in the schema so the model picks rather than invents, and a `model_validator` catches the one incoherent combination — an `update` that does not say what it is updating.

The validator earns its place here rather than decorating the schema. `action="update"` with no `target_id` is a decision the calling code cannot execute — it says *replace something* without saying what — and it is exactly the shape a model produces when it half-commits. Raising sends the real message back and the model fills it in; the alternative is a `None` reaching `journal.update(None, text)`.

Three conclusions, one of each kind:

Measured over three runs each, those three are stable: `ignore`, `update` naming `#21`, and `add`.

A fourth is not, and it is the honest one to show. Offer it *"On 2 July, raising max connections to 60 cleared the pool exhaustion"* — which restates `#1` and adds a date — and it splits between `ignore` and `update #1` across runs.

That is not a bug being papered over. "Is a date worth rewriting an entry for?" has no correct answer, so the model does not have one either. What matters is what it **never** chose: `add`. On the ambiguous case the procedure degrades toward not duplicating, which is the direction you want it to fail in.

### What this actually costs, and what it does not fix

**Every write now costs a retrieve and a model call.** Together with the summary from the previous section, an agent writing a memory makes three round trips before anything is stored. That is the price of a journal that stays consistent, and it is worth knowing before agreeing to it — for a store nothing ever searches, it buys nothing.

**The check is only as good as the search.** This is the part worth dwelling on. `decide()` classifies against whatever `Recall` returned, so a memory the search fails to surface may as well not exist: the model sees nothing resembling the candidate, answers `add`, and the duplicate is created by the same procedure that exists to prevent it.

Which puts the floor from the retrieve section under a second, opposite pressure. For a user's question, returning nothing is a feature — better silence than the least-bad of five bad candidates. For a duplicate check, returning nothing is read as **permission**, and a floor set too high quietly manufactures the duplicates it was meant to catch.

Two consequences worth taking seriously:

- The `k` for a write check and the `k` for answering a question are different numbers. A question wants the best three; a duplicate check wants everything that could plausibly be the same fact, and would rather look at ten.
- The floor for a write check should be lower than the one for recall, or absent. Being shown an irrelevant memory costs the classifier nothing — it answers `add` anyway. Being shown nothing costs correctness.

**And it does not fix a memory that was simply wrong.** Nothing here checks a conclusion against the world; it checks it against what is already stored. An agent confidently wrong twice in the same direction produces a consistent journal that is consistently wrong, and this procedure will happily `ignore` the second one for agreeing with the first.

## `forget`

Two things wear this name, and only one of them removes anything.

**Decay** changes where a memory ranks. The row stays in the table, keeps costing index and storage, and stays eligible for every query.

**Deletion** removes it, and is optional.

Start with what decay looks like across the whole journal. Nothing here has been used, so what is on screen is pure age:

A ladder from about `0.03` for the oldest entries to `0.93` for the newest. `#1` and `#2`, from the first week of July, have decayed to roughly three percent of a fresh memory.

So ask a question that only their neighbourhood can answer:

**A memory at a few percent activation comes back first.**

That is decay working as designed, and it is the thing to be clear about. Decay lowers a *score*. It does not remove a row, and it does not lower that row's standing against memories which are not in the store. When a decayed entry is the closest thing the journal has to a question, a low score is still the highest score.

So decay by itself never makes the store smaller. Everything ever written stays, ranked further and further down. Whether that is a problem is a decision, not a fact.

### Keep everything, or set a bound

`Recall` takes an `evict_below` figure, and it is **`None` by default — nothing is ever deleted for having decayed.** Memories sink in the ranking as they go unused, and stay.

That is a reasonable permanent answer, not just a safe starting one. A journal of a few thousand entries is a couple of megabytes and a search that is still instant; there is nothing worth reclaiming, and anything you keep is something you can still be asked for.

The case for a bound shows up when the store grows faster than the questions do — an agent writing notes every run, for years — where the cost is not disk but **dilution**: every retrieve is a top-k, and a k full of memories nothing has needed since spring is a k that has no room for the ones it has.

Setting `evict_below` turns activation into that bound. `stale()` reports what falls under it:

**At `0.01` it matches nothing at all**, because the oldest memory here has only reached `0.0298`. Five months is not long enough for anything to die at a thirty-day half-life. Like the floor and the weights, the number comes off your own corpus — a store with years in it would evict at `0.01` constantly, and one a fortnight old should evict at nothing.

At `0.05` two entries qualify, and the pair is worth looking at before deleting either.

- **`#1`** — *"raising max connections to 60 cleared it"*. July, never returned to.
- **`#2`** — *"peak traffic on checkout lands between 19:00 and 21:00"*. Also July, also never returned to.

Activation cannot separate them, because it only measures **use**, and neither has been used. But `#2` is the kind of thing nobody asks until the week they need it badly. *Unused* and *not worth keeping* are not the same property, and no threshold can tell them apart.

Which is what `pin()` is for, and why a sweep reports before it deletes:

`sweep()` reports by default and deletes only when told, because there is no undo and the list is the part worth reading twice.

It also has to reach **both** indexes. `Journal.forget` removes the SQLite row and the Chroma vector together — miss one and the entry stays findable by half the questions that should find it, which raises no error and surfaces only as a search that quietly stopped working.

Re-running this notebook restores the journal: the seeding cell checks the count and rebuilds when it does not match. A section that deletes things needs the section above it to be reproducible.

### On a graph, a lifetime instead

`Graph` has no `forget()`, and adding one would be a mistake.

*"Meera owns billing-api"* is not something to remove when Devi takes over on the twentieth of August. It is something that was the case for a period, and the period has ended. Delete the edge and *who owned this in July* becomes unanswerable; overwrite it with Devi and the graph will answer that question confidently and wrongly.

So the edge gets an end date and stays walkable:

Three questions, three correct answers, out of a graph nothing was destructively edited in. July returns Meera, September returns Devi, and `at=None` — *do not filter by time* — returns both, because both describe real periods.

That is the contrast to take out of this section. A journal can only keep a memory or delete it, so bounding it means **losing** what it drops. A graph can say *when* something held, so the same operation costs nothing: the edge stops being current without stopping being available.

Which of those you get is decided by the store, and the store was decided in notebook 2 by what would be asked of it. "Who owns this now" and "who owned this in July" are both questions about the same edge, and only one shape of store can answer both.

## What you built

✓ Separated **read** from **retrieve** — a read is exact and can say no, a retrieve is ranked and cannot — then found that only one of the four stores ranks at all. SQL orders by a column and a graph returns the chain it walked; neither produces a score, and neither is missing one

✓ Timed the same verb across three stores — `1.1 ms`, `4.0 ms`, `733.8 ms` — and hit a fourth that cannot be called from a sentence at all, because `Graph.write` takes a triple and something has to produce it

✓ Found that one sentence is **two** edges: *"Devi took over billing-api from Meera"* starts one and ends another, and a schema with a single list would have left the graph claiming two owners forever

✓ Watched a real memory produce **no** edges — *"the pool ran dry under peak traffic"* is journal entry #1, retrieved twice in notebook 2, and has nothing to walk. The extraction ran, cost a call, and correctly returned nothing

✓ Proved a vector update is a replacement rather than an edit, by watching one entry barely move for a phrase both versions kept (`0.7507 → 0.7392`) and drop hard for one only the original had (`0.7541 → 0.6660`)

✓ Gave `Records` no `update()` on purpose, because an event log's whole value is that you cannot edit it, and `Graph` an `expire()` instead of one

✓ Measured where a floor belongs rather than picking one: answerable questions land at **0.72–0.78**, unanswerable ones at **0.52–0.55**, and that gap lives in the **raw** cosine which min-max destroys — so the only step that can say *I do not have that* has to run before the rescaling

✓ Separated three things that all look like time: **reinforcement** (has this been used), **validity** (is it still true), and the **write date** (metadata to filter on). Only the first is a ranking signal, and confusing it with the second is how a stale fact gets nudged down a list it is still on

✓ Recorded uses in an `accesses` table and watched the same query return a different third place depending on what the agent had been reading — while `#17` and `#9` never moved, because reinforcement breaks ties and does not override relevance

✓ Measured exactly how weak that is: `#18` trails `#14` by 0.02 of cosine and **a hundred uses does not close it**, because activation saturates at `0.9991` and `0.2 × 1.0` is the most the term can ever contribute

✓ Put frequency **inside** lambda rather than beside it — `λ(f) = λ₀/(1+αf)` — so it stretches the clock instead of competing with recency. A memory used sixty times survives seventeen years; a one-off dies in six months

✓ Printed the two terms in separate columns before trusting a weighted sum, and found cosine moving `0.115` against decay's `0.816`, which made "fifty-fifty" seven-eighths recency in practice

✓ Measured the weights instead of choosing them. Six questions with known answers rule out the even split — it gets two of them wrong — and at `w_reinforcement = 0.5` a single memory takes first place on **every** query and collects **every** access in the corpus

✓ Built two-stage recall only after showing it should not exist for one-line memories, where the summary *is* the entry, then compressed six session write-ups about nine to one and let the model open what it wanted

✓ Fixed four things that looked right and were not: a loop missing its trailing `Observe()`, so tool results never reached the model; a listing with no dates, so a question about November opened October; a docstring that never said what ids were for; and a demo question the summaries already answered

✓ Derived `record_use` from the store rather than leaving it a flag — **the last stage records** — because a single-stage store with nothing recording has an empty `accesses` table and a reinforcement term that has quietly become write-date decay

✓ Gave the write path a decision — **add / update / ignore** — so the agent searches before it writes and `update` finally has something that calls it, with a `model_validator` rejecting an `update` that names nothing to update

✓ Said plainly what that check cannot do: it is bounded by the search, so a memory retrieval fails to surface becomes a duplicate created by the very procedure meant to prevent it — and it compares a conclusion against the store, never against the world

✓ Separated decay from deletion, and watched a memory at `0.0351` activation still come back **first**, because decay lowers a score and does not remove a row

✓ Made eviction a policy that is **off by default**, then found `0.01` matches nothing on a five-month journal while `0.05` catches one entry worth losing and one that is merely never asked about — which is what `pin()` and a dry run are for

✓ Ended on the graph, where forgetting is neither decay nor deletion: an edge gets an end date, so July returns Meera, September returns Devi, and `at=None` returns both

✓ Left three numbers explicitly unearned — `H₀ = 30 days`, `α = 0.5`, and the linear-versus-damped choice — because a journal spanning five months cannot separate them. Every access is recent next to either half-life, so the curves overlap. They need a corpus with years in it, and this one has months

**Next:** `04_types_of_memory.ipynb` — semantic, episodic and procedural memory: which store each belongs in, and what breaks when they are paired wrong. The write path is where they diverge hardest, and the decision built here is the example: **add / update / ignore** is right for a fact and destroys a run history, because *"deploy failed on the 3rd"* and *"deploy failed on the 12th"* look like duplicates to a similarity check and are two events.